In [1]:
## Importing Libraries
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
import torch

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'CPU')
print(f'Using Device : {device}')

Using Device : cuda


In [4]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
df.shape

(60000, 785)

In [6]:
# train test split
from sklearn.model_selection import train_test_split
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Scaling the features
X_train = X_train/255
X_test = X_test/255

In [8]:
## Creating CustomDataset Class
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype= torch.float32).reshape(-1,1,28,28)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [9]:
## Creating train dataset object
train_dataset = CustomDataset(X_train, y_train)
len(train_dataset)

48000

In [10]:
## Creating custom test dataset
test_dataset = CustomDataset(X_test, y_test)
len(test_dataset)

12000

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [12]:
from torch.nn.modules.dropout import Dropout
class CNN(nn.Module):
  def __init__(self, input_features):
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(in_channels=input_features, out_channels=32, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=64*7*7, out_features=128),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(64,10)
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [13]:
## Learning Rate and Epochs
learning_rate = 0.1
epochs = 100

In [14]:
## Instanciate the Model
model = CNN(1)
model.to(device=device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [15]:
## Training Loop
for epoch in range(epochs):
    model.train()
    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:
        # Move inputs and labels to the same device as the model
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)

        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    print(f'Epoch: {epoch + 1}, Loss: {avg_loss:.4f}')

Epoch: 1, Loss: 0.5854
Epoch: 2, Loss: 0.4076
Epoch: 3, Loss: 0.3500
Epoch: 4, Loss: 0.3090
Epoch: 5, Loss: 0.2818
Epoch: 6, Loss: 0.2638
Epoch: 7, Loss: 0.2465
Epoch: 8, Loss: 0.2281
Epoch: 9, Loss: 0.2163
Epoch: 10, Loss: 0.2055
Epoch: 11, Loss: 0.1942
Epoch: 12, Loss: 0.1855
Epoch: 13, Loss: 0.1844
Epoch: 14, Loss: 0.1759
Epoch: 15, Loss: 0.1659
Epoch: 16, Loss: 0.1643
Epoch: 17, Loss: 0.1574
Epoch: 18, Loss: 0.1561
Epoch: 19, Loss: 0.1502
Epoch: 20, Loss: 0.1453
Epoch: 21, Loss: 0.1430
Epoch: 22, Loss: 0.1390
Epoch: 23, Loss: 0.1350
Epoch: 24, Loss: 0.1331
Epoch: 25, Loss: 0.1269
Epoch: 26, Loss: 0.1232
Epoch: 27, Loss: 0.1194
Epoch: 28, Loss: 0.1138
Epoch: 29, Loss: 0.1135
Epoch: 30, Loss: 0.1118
Epoch: 31, Loss: 0.1098
Epoch: 32, Loss: 0.1088
Epoch: 33, Loss: 0.1076
Epoch: 34, Loss: 0.1046
Epoch: 35, Loss: 0.1005
Epoch: 36, Loss: 0.1041
Epoch: 37, Loss: 0.0993
Epoch: 38, Loss: 0.0938
Epoch: 39, Loss: 0.0932
Epoch: 40, Loss: 0.0912
Epoch: 41, Loss: 0.0958
Epoch: 42, Loss: 0.0959
E

In [16]:
## Model Evaluation
model.eval()

CNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [17]:
## Accuracy on Test Data
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    # Move batches to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total = total + batch_labels.shape[0]
    correct = correct + (predicted == batch_labels).sum().item()

  print(correct / total)

0.9181666666666667


In [18]:
## Accuracy on Training Data
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    # Move batches to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total = total + batch_labels.shape[0]
    correct = correct + (predicted == batch_labels).sum().item()

  print(correct / total)

0.9952916666666667
